# Intermediate 03 — Law of Large Numbers Revisited

Practice notebook for the **"Law of Large Numbers Revisited"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
# Part 1 — Weak Law of Large Numbers (WLLN)

The PDF restates the **Weak Law of Large Numbers**:

> Let \(X_1, X_2, \dots\) be i.i.d. with \(E[X_i] = \mu\) and \(\operatorname{Var}(X_i) = \sigma^2 < \infty\). Then
> $$\bar{X}_n \to \mu.$$

The arrow \(\to\) means **convergence in probability**: for every \(\varepsilon > 0\),

$$P\!\left(\left|\bar{X}_n - \mu\right| > \varepsilon\right) \to 0 \quad \text{as } n \to \infty.$$

The proof uses **Chebyshev's inequality**: since \(E[\bar{X}_n] = \mu\) and \(\operatorname{Var}(\bar{X}_n) = \sigma^2/n\),

$$P\!\left(\left|\bar{X}_n - \mu\right| \geq \varepsilon\right) \leq \frac{\sigma^2}{n\,\varepsilon^2}.$$

The right-hand side goes to zero, so the probability on the left must too. The key mechanism: the sample mean's variance shrinks like \(1/n\), so it is increasingly concentrated near \(\mu\).

Below we **estimate** \(P(|\bar{X}_n - \mu| > \varepsilon)\) directly from many independent replicates and watch it fall toward 0 as \(n\) grows.


In [ ]:
# WLLN in action: estimate P(|Xbar_n - mu| > eps) over many replicates
rng = np.random.default_rng(42)

mu, sigma = 2.0, 1.5          # parent: Normal(2, 1.5^2)
eps = 0.10                     # tolerance
n_replicates = 20_000          # independent sequences per n
n_values = [10, 50, 100, 500, 1_000, 5_000, 20_000]

chebyshev = []
empirical = []
for n in n_values:
    # Each row is a replicate; take the sample mean across columns
    draws = rng.normal(loc=mu, scale=sigma, size=(n_replicates, n))
    xbar = draws.mean(axis=1)
    p_emp = np.mean(np.abs(xbar - mu) > eps)
    p_cheb = sigma**2 / (n * eps**2)
    empirical.append(p_emp)
    chebyshev.append(p_cheb)
    print(f"n={n:6d}: P(|Xbar-mu|>eps) empirical = {p_emp:.4f}, "
          f"Chebyshev bound = {min(p_cheb, 1.0):.4f}")

fig, ax = plt.subplots()
ax.plot(n_values, empirical, "o-", label=r"empirical  $P(|\bar{X}_n-\mu|>\varepsilon)$")
ax.plot(n_values, [min(c, 1.0) for c in chebyshev], "s--", color="tab:red",
        label=r"Chebyshev  $\sigma^2/(n\varepsilon^2)$")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("sample size  $n$"); ax.set_ylabel("probability")
ax.set_title(rf"WLLN: $P(|\bar{{X}}_n-\mu|>\varepsilon)$ with $\varepsilon={eps}$, "
             rf"parent $N({mu},{sigma}^2)$")
ax.legend(); plt.show()


**Your turn:** Halve \(\varepsilon\) to 0.05. How much larger must \(n\) be to reach the same empirical probability as before? Does this match the \(1/\varepsilon^2\) dependence of the Chebyshev bound?


---
# Part 2 — Strong Law of Large Numbers (SLLN)

The PDF then states the **Strong Law of Large Numbers** (informal):

> Under mild conditions (for example, i.i.d. with finite expectation \(E[|X_i|] < \infty\)), we have
> $$\bar{X}_n \to} \mu.$$

This is **stronger** than the WLLN: for *almost every* realization, the time average converges to the expected value. Intuitively, if you keep rolling a fair die forever, the long-run average outcome **for a single sequence of rolls** will be 3.5.

The distinction:

- **WLLN** fixes \(n\) and asks about the *distribution* of \(\bar{X}_n\) across many replications → probability of a large deviation goes to 0.
- **SLLN** fixes a *single infinite sequence* and asks whether its running average converges → it does, with probability 1.

We visualize the SLLN by plotting several independent long sequences and watching each one settle onto \(\mu\).


In [ ]:
# SLLN visual: many independent long sequences, each should settle on mu
rng = np.random.default_rng(7)
mu, sigma = 2.0, 1.5
n_max = 20_000
n_paths = 12

# Each row is an independent sequence; plot its running mean
draws = rng.normal(loc=mu, scale=sigma, size=(n_paths, n_max))
running = np.cumsum(draws, axis=1, dtype=float) / np.arange(1, n_max + 1)

fig, ax = plt.subplots()
n_grid = np.arange(1, n_max + 1)
for i in range(n_paths):
    ax.plot(n_grid, running[i], lw=0.7, alpha=0.75)
ax.axhline(mu, color="black", ls="--", lw=1.5, label=r"$\mu = 2.0$")
# 2-sigma shrinking envelope: typical spread is sigma/sqrt(n)
ax.fill_between(n_grid, mu - 2*sigma/np.sqrt(n_grid),
                mu + 2*sigma/np.sqrt(n_grid),
                color="tab:gray", alpha=0.18, label=r"$\mu \pm 2\sigma/\sqrt{n}$")
ax.set_xscale("log")
ax.set_xlabel("sample size  $n$"); ax.set_ylabel(r"running mean  $\bar{X}_n$")
ax.set_title(r"SLLN — each path's running mean converges to $\mu$")
ax.legend(); plt.show()

# How many of the 12 paths are within eps of mu at n = n_max?
eps = 0.05
close = np.sum(np.abs(running[:, -1] - mu) < eps)
print(f"At n={n_max}: {close}/{n_paths} paths within eps={eps} of mu={mu}")


**Your turn:** Add a 13th path drawn from a distribution with the same mean but much larger variance (e.g. `sigma=10`). Does the running mean still converge, and how does its *speed* compare to the others?


---
# Part 3 — WLLN vs SLLN: Two Viewpoints on the Same Object

WLLN and SLLN are two statements about the *same* sample mean \(\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i\):

- **WLLN** (convergence in probability): for a fixed large \(n\), if we draw *many* independent copies of \(\bar{X}_n\), the fraction that fall outside \((\mu-\varepsilon, \mu+\varepsilon)\) goes to 0.
- **SLLN** (almost-sure convergence): for a single infinite sequence \(X_1, X_2, \dots\), the running average stays within \((\mu-\varepsilon, \mu+\varepsilon)\) eventually and forever (with probability 1).

Almost-sure convergence implies convergence in probability, but not vice versa. Here we compute, for a grid of \(n\), both the **empirical deviation probability** (WLLN view, many replications) and the **fraction of single-sequence paths still outside** the band at \(n\) (a finite-\(n\) proxy for the SLLN view). They agree closely because we are estimating the same underlying quantity from two angles.


In [ ]:
# Compare the two viewpoints numerically
rng = np.random.default_rng(2024)
mu, sigma = 2.0, 1.5
eps = 0.10
n_values = [50, 200, 1_000, 5_000, 20_000]
n_paths = 5_000           # independent sequences tracked over time (SLLN view)
n_max = n_values[-1]

# Track running means of n_paths independent sequences
draws = rng.normal(loc=mu, scale=sigma, size=(n_paths, n_max))
cum_sum = np.cumsum(draws, axis=1, dtype=float)
grid = np.arange(1, n_max + 1)
running = cum_sum / grid

print(f"{'n':>7}  {'WLLN view':>12}  {'SLLN view':>12}  {'Chebyshev':>12}")
for n in n_values:
    # WLLN view: fresh draws at exactly this n (independent of the running paths)
    fresh = rng.normal(loc=mu, scale=sigma, size=(n_paths, n)).mean(axis=1)
    p_wlln = np.mean(np.abs(fresh - mu) > eps)
    # SLLN view: fraction of long-running paths whose value AT index n is outside the band
    p_slln = np.mean(np.abs(running[:, n-1] - mu) > eps)
    p_cheb = min(sigma**2 / (n * eps**2), 1.0)
    print(f"{n:7d}  {p_wlln:12.4f}  {p_slln:12.4f}  {p_cheb:12.4f}")

# Plot a few example paths with the eps-band
fig, ax = plt.subplots()
for i in range(8):
    ax.plot(grid, running[i], lw=0.6, alpha=0.7)
ax.axhline(mu, color="black", ls="--", lw=1.2)
ax.fill_between(grid, mu - eps, mu + eps, color="tab:green", alpha=0.15,
                label=rf"$\mu \pm \varepsilon$, $\varepsilon={eps}$")
ax.set_xscale("log")
ax.set_xlabel("sample size  $n$"); ax.set_ylabel(r"$\bar{X}_n$")
ax.set_title("WLLN vs SLLN — same paths, two convergence notions")
ax.legend(); plt.show()


**Your turn:** Why do the two empirical columns agree so well? Explain in one sentence why tracking \(P(|\bar{X}_n - \mu| > \varepsilon)\) at a single fixed \(n\) is *not* the same as the SLLN statement, even though the numbers match here.


---
# Part 4 — When the LLN Fails: the Cauchy Distribution

Both laws require \(E[|X_i|] < \infty\). The PDF's SLLN statement explicitly mentions this condition. The classic counterexample is the **Cauchy** distribution, whose density is

$$f(x) = \frac{1}{\pi(1+x^2)},$$

which has *no finite mean* — the integral \(E[|X|]\) diverges. For Cauchy samples the sample mean \(\bar{X}_n\) has the **same** distribution as a single observation, so it does **not** converge to anything. The LLN genuinely fails.

We contrast Normal(0,1) (finite mean, LLN holds) with Cauchy(0,1) (no mean, LLN fails) by plotting running means of several independent sequences for each.


In [ ]:
# Normal vs Cauchy: LLN holds vs LLN fails
rng = np.random.default_rng(11)
n_max = 50_000
n_paths = 6
grid = np.arange(1, n_max + 1)

# Normal(0,1): mean 0, variance 1 -> LLN holds
normal_draws = rng.standard_normal(size=(n_paths, n_max))
normal_running = np.cumsum(normal_draws, axis=1, dtype=float) / grid

# Cauchy(0,1): no mean -> LLN fails
cauchy_draws = rng.standard_cauchy(size=(n_paths, n_max))
cauchy_running = np.cumsum(cauchy_draws, axis=1, dtype=float) / grid

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
for i in range(n_paths):
    axes[0].plot(grid, normal_running[i], lw=0.7, alpha=0.75)
axes[0].axhline(0.0, color="black", ls="--", lw=1.2, label=r"$\mu = 0$")
axes[0].set_xscale("log")
axes[0].set_xlabel("sample size  $n$"); axes[0].set_ylabel(r"running mean  $\bar{X}_n$")
axes[0].set_title("Normal(0,1) — LLN holds: running mean -> 0")
axes[0].legend()

for i in range(n_paths):
    axes[1].plot(grid, cauchy_running[i], lw=0.7, alpha=0.75)
axes[1].axhline(0.0, color="black", ls="--", lw=1.2)
axes[1].set_xscale("log")
axes[1].set_xlabel("sample size  $n$"); axes[1].set_ylabel(r"running mean  $\bar{X}_n$")
axes[1].set_title("Cauchy(0,1) — LLN fails: running mean wanders")
plt.tight_layout(); plt.show()

# Numerical sanity check: spread of running means at n = n_max
print(f"Normal  at n={n_max}: running means range "
      f"[{normal_running[:,-1].min():.4f}, {normal_running[:,-1].max():.4f}]")
print(f"Cauchy  at n={n_max}: running means range "
      f"[{cauchy_running[:,-1].min():.2f}, {cauchy_running[:,-1].max():.2f}]")


**Your turn:** Replace `rng.standard_cauchy` with `rng.standard_t(df=2)`, which has a finite mean but infinite variance. Does the running mean converge? Does it converge *faster* or *slower* than the Normal case, and why?


---
# Part 5 — PDF Example: Simulated Trading P&L

The PDF applies the SLLN to a trading strategy. Suppose daily profit and loss \(X_1, X_2, \dots\) is i.i.d. with \(E[X_i] = \mu\). The strong law says that for a single long simulation path, the average P&L

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$$

approaches \(\mu\) almost surely. This justifies treating expected daily P&L as the long-run average performance in a stable market regime.

We model daily P&L as \(X_i = \mu + \sigma Z_i\) with \(Z_i \sim N(0,1)\), a simple but common baseline. We simulate a few years of trading days and watch the annualized running average P&L converge to \(\mu\).


In [ ]:
# Trading P&L: SLLN on a single long simulation path
rng = np.random.default_rng(2025)
mu_daily, sigma_daily = 0.0005, 0.010       # ~5 bps/day mean, 1% daily vol
days = 252 * 20                               # 20 years of trading days
n_paths = 5

fig, ax = plt.subplots()
for k in range(n_paths):
    pnl = mu_daily + sigma_daily * rng.standard_normal(size=days)
    running_avg = np.cumsum(pnl, dtype=float) / np.arange(1, days + 1)
    ax.plot(np.arange(1, days + 1), running_avg, lw=0.8, alpha=0.8,
            label=f"path {k+1}" if k < 3 else None)
ax.axhline(mu_daily, color="black", ls="--", lw=1.3,
           label=rf"$\mu = {mu_daily:.4f}$")
ax.set_xscale("log")
ax.set_xlabel("trading day  $n$"); ax.set_ylabel(r"running average P&L  $\bar{X}_n$")
ax.set_title(r"SLLN — running average daily P&L converges to $\mu$")
ax.legend(); plt.show()

# Summary table of running averages at selected horizons
pnl = mu_daily + sigma_daily * rng.standard_normal(size=days)
running_avg = np.cumsum(pnl, dtype=float) / np.arange(1, days + 1)
for yr in [1, 5, 10, 20]:
    n = 252 * yr
    print(f"After {yr:2d} yr (n={n:5d}): running avg P&L = {running_avg[n-1]:.5f}  "
          f"(theory {mu_daily:.5f})")


**Your turn:** Set `mu_daily = 0.0` (a strategy with no edge). Re-run the cell and describe what the running average P&L does over 20 years. Why is this the most important case to understand before risking capital?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State the **Weak** and **Strong** Laws of Large Numbers and explain in one or two sentences why the strong law is a *stronger* statement than the weak law. Which condition on the distribution of \(X_i\) does each require?

2. Using Chebyshev's inequality, derive the bound \(P(|\bar{X}_n - \mu| > \varepsilon) \leq \sigma^2/(n\varepsilon^2)\). For \(X_i \sim N(0, 4)\) (so \(\sigma^2 = 4\)) and \(\varepsilon = 0.1\), compute the Chebyshev bound at \(n = 100, 1000, 10000\) and compare it to a simulation estimate of the same probability with 50,000 replications.

3. Simulate 20 independent Cauchy(0,1) sequences of length 100,000 and plot their running means on a log-x axis. Explain in your own words why the plot looks nothing like the Normal case and what this says about the necessity of the condition \(E[|X_i|] < \infty\) in the SLLN.

4. For \(X_i \sim \text{Exponential}(\lambda=1)\) with \(\mu = 1\) and \(\sigma^2 = 1\), simulate a single path of length 200,000 and report \(\bar{X}_n\) at \(n = 10, 100, 1000, 10000, 100000, 200000\). Confirm the gap \(|\bar{X}_n - \mu|\) shrinks roughly like \(\sigma/\sqrt{n}\) and discuss whether your single path is consistent with the SLLN.

5. A trading strategy has daily P&L \(X_i = 0.0008 + 0.012 Z_i\) with \(Z_i \sim N(0,1)\). Using the SLLN, argue why the long-run average daily P&L of a single simulation path converges to 0.0008. Simulate 10 paths of 10 years (2520 days each) and report the range of final running averages across the paths.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** **WLLN:** if \(X_i\) are i.i.d. with \(E[X_i]=\mu\) and \(\sigma^2 < \infty\), then \(\bar{X}_n \to \mu\), i.e. for every \(\varepsilon>0\), \(P(|\bar{X}_n-\mu|>\varepsilon)\to 0\). **SLLN:** if \(X_i\) are i.i.d. with \(E[|X_i|]<\infty\), then \(\bar{X}_n \to} \mu\). The strong law is stronger because almost-sure convergence implies convergence in probability but not vice versa: it makes a statement about a *single infinite sequence* (the running average eventually stays near \(\mu\) forever), whereas the weak law only controls the *distribution* of \(\bar{X}_n\) at each fixed \(n\). The WLLN requires finite variance; the SLLN only requires finite first moment.

**2.** By Chebyshev, \(P(|\bar{X}_n-\mu|>\varepsilon) \leq \sigma^2/(n\varepsilon^2)\) because \(\operatorname{Var}(\bar{X}_n)=\sigma^2/n\). With \(\sigma^2=4\), \(\varepsilon=0.1\):

```python
import numpy as np
rng = np.random.default_rng(0)
mu, sigma2, eps = 0.0, 4.0, 0.1
for n in [100, 1000, 10000]:
    draws = rng.normal(mu, np.sqrt(sigma2), size=(50_000, n))
    p_emp = np.mean(np.abs(draws.mean(axis=1) - mu) > eps)
    p_cheb = min(sigma2/(n*eps**2), 1.0)
    print(f"n={n:5d}: empirical={p_emp:.4f}, Chebyshev={p_cheb:.4f}")
```
Chebyshev is a loose upper bound (it does not use the normal shape), so the empirical probability is much smaller than the bound, but both shrink like \(1/n\).

**3.**
```python
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(1)
n_max, n_paths = 100_000, 20
grid = np.arange(1, n_max + 1)
running = np.cumsum(rng.standard_cauchy(size=(n_paths, n_max)), axis=1) / grid
fig, ax = plt.subplots()
for i in range(n_paths):
    ax.plot(grid, running[i], lw=0.6, alpha=0.7)
ax.axhline(0, color="k", ls="--")
ax.set_xscale("log"); ax.set_xlabel("n"); ax.set_ylabel(r"$\bar{X}_n$")
ax.set_title("Cauchy(0,1) — LLN fails"); plt.show()
```
The running means wander forever and occasionally jump by large amounts because a single huge Cauchy draw shifts the average by a lot even at large \(n\). Cauchy has \(E[|X|]=\infty\), so the SLLN hypothesis is violated and there is no number for \(\bar{X}_n\) to converge to.

**4.**
```python
import numpy as np
rng = np.random.default_rng(2)
mu, sigma = 1.0, 1.0
n_max = 200_000
x = rng.exponential(1.0, size=n_max)
running = np.cumsum(x, dtype=float) / np.arange(1, n_max + 1)
for n in [10, 100, 1000, 10000, 100000, 200000]:
    gap = abs(running[n-1] - mu)
    print(f"n={n:7d}: Xbar={running[n-1]:.5f}, gap={gap:.4f}, "
          f"sigma/sqrt(n)={sigma/np.sqrt(n):.4f}")
```
The gap is of order \(\sigma/\sqrt{n}\) and decreases steadily, so this single path behaves as the SLLN predicts: it stays close to \(\mu=1\) and the deviations shrink with \(n\).

**5.** By the SLLN, since \(E[X_i] = 0.0008 < \infty\), the single-path running average \(\bar{X}_n \to} 0.0008\). The long-run average daily P&L of any one simulation path therefore converges to the strategy's expected daily P&L.

```python
import numpy as np
rng = np.random.default_rng(3)
mu_daily, sigma_daily, days = 0.0008, 0.012, 2520
finals = []
for _ in range(10):
    pnl = mu_daily + sigma_daily * rng.standard_normal(days)
    finals.append(pnl.mean())
finals = np.array(finals)
print(f"range of final running averages: [{finals.min():.5f}, {finals.max():.5f}]")
print(f"theory mu = {mu_daily:.5f}")
```
The 10 final averages cluster tightly around 0.0008, with spread of order \(\sigma/\sqrt{2520}\), confirming the SLLN prediction.

</details>
